In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from skimage.measure import label, regionprops
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
import cartopy.crs as ccrs
from matplotlib.path import Path

In [ ]:


ds = xr.open_dataset("data/SLP_ex.nc")
da = ds["SLP"]

threshold = 100000
min_area = 20
min_overlap = 0.2  



# ---- get lat/lon coords ----
# assumes dims like (time, lat, lon)
lat = da[da.dims[1]].values
lon = da[da.dims[2]].values

# if lat/lon are 1D, make 2D grids for contourf/contour
lon2d, lat2d = np.meshgrid(lon, lat)


In [ ]:
def track_slp_features_matplotlib(
    da,
    threshold=100000,
    min_area=20,
    min_overlap=0.2,
):
    """
    Track SLP low-pressure features using Matplotlib contours.

    Feature definition:
        SLP <= threshold

    Tracking rule:
        carry track ID forward if current feature overlaps a previous feature.
    """

    ny, nx = da.isel(time=0).shape
    yy, xx = np.mgrid[0:ny, 0:nx]
    points = np.column_stack([xx.ravel(), yy.ravel()])

    tracked = []
    prev = {}
    next_track_id = 1

    for t in range(da.sizes["time"]):

        field = da.isel(time=t).values

        fig, ax = plt.subplots()
        cs = ax.contour(field, levels=[threshold])
        plt.close(fig)

        current = {}
        frame_info = []

        for seg in cs.allsegs[0]:

            path = Path(seg)
            mask = path.contains_points(points).reshape(ny, nx)

            # keep low-pressure side only
            mask = mask & (field <= threshold)

            if mask.sum() < min_area:
                continue

            y, x = np.where(mask)
            centroid = (y.mean(), x.mean())

            best_id = None
            best_overlap = 0

            for track_id, old_mask in prev.items():
                overlap = np.logical_and(mask, old_mask).sum() / mask.sum()

                if overlap > best_overlap:
                    best_overlap = overlap
                    best_id = track_id

            if best_overlap < min_overlap:
                best_id = next_track_id
                next_track_id += 1

            current[best_id] = mask

            frame_info.append({
                "track_id": best_id,
                "mask": mask,
                "centroid_rc": centroid,
            })

        tracked.append(frame_info)
        prev = current

    return tracked

In [ ]:
tracked_mpl = track_slp_features_matplotlib(
    da,
    threshold=threshold,
    min_area=20,
    min_overlap=0.2,
)

In [ ]:
lat = da[da.dims[1]].values
lon = da[da.dims[2]].values
lon2d, lat2d = np.meshgrid(lon, lat)

levels = np.linspace(float(da.min()), float(da.max()), 16)

fig, ax = plt.subplots(
    figsize=(8, 4),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

def update(t):
    ax.clear()

    field = da.isel(time=t).values

    # colored SLP contour lines
    ax.contour(
        lon2d, lat2d, field,
        levels=levels,
        cmap="turbo",
        linewidths=0.9,
        transform=ccrs.PlateCarree()
    )

    # tracked feature outlines and IDs
    for feat in tracked_mpl[t]:
        mask = feat["mask"]

        ax.contour(
            lon2d, lat2d, mask.astype(int),
            levels=[0.5],
            colors="black",
            linewidths=1.6,
            transform=ccrs.PlateCarree()
        )

        y, x = feat["centroid_rc"]
        iy = int(round(y))
        ix = int(round(x))

        ax.plot(
            lon[ix], lat[iy],
            "ko", ms=3,
            transform=ccrs.PlateCarree()
        )

        ax.text(
            lon[ix], lat[iy],
            str(feat["track_id"]),
            fontsize=8,
            color="black",
            ha="left",
            va="bottom",
            transform=ccrs.PlateCarree(),
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1)
        )

    ax.coastlines(color="0.55", linewidth=0.6)

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="0.5",
        alpha=0.25,
        linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False

    ax.set_title(f"Matplotlib contour-tracked SLP features | time index = {t}")

anim = FuncAnimation(
    fig,
    update,
    frames=da.sizes["time"],
    interval=300,
    blit=False,
)

plt.close(fig)
HTML(anim.to_jshtml())